# Big-Data Project - PulseLM Biosignal Pipeline

**Dataset:** [`Manhph2211/PulseLM`](https://huggingface.co/datasets/Manhph2211/PulseLM)

## Index

- [0: Environment setup](#0-environment-setup)
- [1: Feature extractor](#1-feature-extractor)
  - [1.1: QA label parser](#11-qa-label-parser)
  - [1.2: Diagnostic - preview the QA format](#12-diagnostic---preview-the-qa-format)
- [2: Streaming ingestion → parquet files with labels](#2-streaming-ingestion--parquet-files-with-labels)
- [3: Load features back from disk (lazy, low-memory)](#3-load-features-back-from-disk-lazy-low-memory)
- [4: Load parquet with Spark for distributed processing](#4-load-parquet-with-spark-for-distributed-processing)
  - [4.1: Data cleaning: schema check, null/NaN filter, SQI gate](#41-data-cleaning-schema-check-nullnan-filter-sqi-gate)
  - [4.2: Label coverage and class distribution](#43-label-coverage-and-class-distribution)
  - [4.3: Example signals from each dataset](#44-example-signals-from-each-dataset)
- [5: Machine learning pipeline](#5-machine-learning-pipeline)
  - [5.1: Prepare binary target columns](#51-prepare-binary-target-columns)
  - [5.2: Spark Random Forest - subset training](#52-spark-random-forest---subset-training)
  - [5.3: Spark Random Forest - whole-dataset training](#53-spark-random-forest---whole-dataset-training)
  - [5.4: Spark MLP - per-task binary classifier](#54-spark-mlp---per-task-binary-classifier)
  - [5.5: Comparison - three models + baseline](#55-comparison---three-models--baseline)
  - [5.6: Demo - clinical Q&A from a signal](#56-demo---clinical-qa-from-a-signal)
  - [5.7: Runtime benchmark - per-phase timings](#57-runtime-benchmark---per-phase-timings)

> ### Dataproc setup notes
>
> 1. **`OUT_DIR`** (section 2) must point at your GCS bucket - replace `YOUR_BUCKET_NAME` before running section 2.
> 2. **Spark session** (section 4) auto-detects Dataproc and picks up the existing YARN session.
> 3. **`CLUSTER_LABEL`** (section 5.7) - set this env var to tag each cluster run (e.g. `dataproc_2w`, `dataproc_4w`). The benchmark cell uses it to label the saved JSON.
>
> **Scaling benchmark workflow:** run on 2-worker cluster → save JSON → resize to 4 workers → restart kernel → re-run section 1.1/section 3/section 4/section 5.1–5.4/section 5.7. Skip section 2 on re-runs - parquet is already on GCS.

## 0: Environment setup

This section installs the libraries and sets the basic paths. When running this in Colab, this is also where you decide whether to mount Google Drive so your outputs survive after the session ends.

In [ ]:
!pip install -q huggingface_hub datasets pyarrow gcsfs scikit-learn matplotlib seaborn tqdm

In [ ]:
from huggingface_hub import login
login()

## 1: Feature extractor

Here we turn each raw signal into a small set of summary numbers. The goal is not to keep every sample, but to capture the parts that are most useful for quick modeling and comparison.

In [ ]:
import numpy as np

FEATURE_NAMES = [
    "mean", "std", "min", "max", "p2p",
    "skew", "kurt", "zcr", "dom_freq",
]

def extract_features(signal):
    """Reduce a 1-D signal array to 9 summary features."""
    if signal is None or len(signal) == 0:
        return [0.0] * len(FEATURE_NAMES)
    x = np.asarray(signal, dtype=np.float32) 
    x = x[np.isfinite(x)]
    if x.size == 0:
        return [0.0] * len(FEATURE_NAMES)

    mean = float(x.mean()); std = float(x.std())
    centered = x - mean
    skew = float(((centered ** 3).mean() / (std ** 3 + 1e-12)))
    kurt = float(((centered ** 4).mean() / (std ** 4 + 1e-12)) - 3.0)
    zcr  = float(np.mean(np.diff(np.signbit(centered)).astype(float)))

    spec = np.abs(np.fft.rfft(centered))
    dom  = float(np.argmax(spec[1:]) + 1) if spec.size > 1 else 0.0

    return [mean, std, float(x.min()), float(x.max()),
            float(x.max() - x.min()), skew, kurt, zcr, dom]

## 1.1: QA label parser

The dataset stores the labels in a QA field, but not always in the same shape. This parser cleans that up, matches each question to the right task, and returns a consistent label name that the rest of the notebook can use.

It also checks answers against the allowed label set so we do not accidentally carry forward malformed text.

In [ ]:
import json, ast

# 12 PulseLM QA tasks
LABEL_TASKS = [
    "HR", "BP", "SQI", "Stress", "SDB",
    "HRV_SDNN", "HRV_RMSSD", "HRV_pNN50",
    "AF", "Arrhythmia", "SpO2", "RR",
]

# Closed-vocabulary answer sets per task 
ANSWER_SETS = {
    "HR":         {"bradycardia", "normal", "tachycardia"},
    "BP":         {"normal", "elevated", "hypertension_stage1", "hypertension_stage2", "hypertensive_crisis"},
    "SQI":        {"good", "noisy"},
    "Stress":     {"baseline", "stress", "amusement", "meditation"},
    "SDB":        {"normal_ahi<5", "mild_5<=ahi<15", "moderate_15<=ahi<30", "severe_ahi>=30"},
    "HRV_SDNN":   {"low", "normal", "high"},
    "HRV_RMSSD":  {"low", "normal", "high"},
    "HRV_pNN50":  {"low", "normal", "high"},
    "AF":         {"af", "non_af"},
    "Arrhythmia": {"sinus_rhythm", "pvc", "pac", "vt", "svt", "af"},
    "SpO2":       {"normal", "abnormal"},
    "RR":         {"bradypnea", "normal", "tachypnea"},
}

# Map QA field keys to task names
TASK_KEY_MAP = {
    "heart_rate_category":    "HR",
    "hr_category":            "HR",
    "blood_pressure_category":"BP",
    "bp_category":            "BP",
    "sqi_category":           "SQI",
    "signal_quality_category":"SQI",
    "stress_label":           "Stress",
    "stress_category":        "Stress",
    "sdb_label":              "SDB",
    "sdb_category":           "SDB",
    "hrv_sdnn_category":      "HRV_SDNN",
    "sdnn_category":          "HRV_SDNN",
    "hrv_rmssd_category":     "HRV_RMSSD",
    "rmssd_category":         "HRV_RMSSD",
    "hrv_pnn50_category":     "HRV_pNN50",
    "pnn50_category":         "HRV_pNN50",
    "af_label":               "AF",
    "af_category":            "AF",
    "arrhythmia_category":    "Arrhythmia",
    "arrhythmia_label":       "Arrhythmia",
    "spo2_category":          "SpO2",
    "rr_category":            "RR",
    "respiratory_rate_category":"RR",
}

# Keyword matching on question text
ROUTER_KEYWORDS = [
    ("HRV_SDNN",   ["sdnn"]),
    ("HRV_RMSSD",  ["rmssd", "parasympathetic"]),
    ("HRV_pNN50",  ["pnn50"]),
    ("AF",         ["atrial fibrillation", "af detection", "af label", "af or non-af",
                    "indicate af", "af present", " af?", " af "]),
    ("Arrhythmia", ["arrhythmia", "cardiac rhythm", "type of arrhythmia",
                    "rhythm classification", "rhythm diagnosis",
                    "heart rhythm abnormality", "rhythm category"]),
    ("SDB",        ["sleep-disordered", "sleep disordered", "sleep apnea",
                    "apnea", "ahi", "sdb", "respiratory disturbance",
                    "sleep breathing", "breathing disorder"]),
    ("Stress",     ["stress", "emotional state", "affective state",
                    "psychological state", "stress condition", "stress state"]),
    ("SQI",        ["signal quality", "sqi", "clean or motion distorted",
                    "good or poor quality", "quality of this ppg",
                    "quality category"]),
    ("BP",         ["blood pressure", "hypertension", "bp class",
                    "bp classification", "bp category"]),
    ("SpO2",       ["spo2", "oxygen saturation", "hypoxemia", "blood oxygen"]),
    ("RR",         ["respiratory rate", "breathing rate", "respiration rate"]),
    ("HR",         ["heart rate", "hr category", "bradycardic", "tachycardic",
                    "pulse rate", "bpm"]),
]


def _norm_answer(s):
    """Normalize answer strings: lowercase, spaces->underscores, dashes->underscores."""
    return str(s).strip().lower().replace(" ", "_").replace("-", "_")


def _decode_qa(qa_field):
    """If qa_field is a string, JSON-decode (or ast literal_eval) it. Else return as-is."""
    if qa_field is None:
        return None
    if isinstance(qa_field, str):
        s = qa_field.strip()
        if not s:
            return None
        try:
            return json.loads(s)
        except json.JSONDecodeError:
            try:
                import ast
                return ast.literal_eval(s)
            except Exception:
                return None
    return qa_field


def _route_by_question(question, answer, validate=True):
    """Fallback router using question-text keywords. Returns (task, answer) or None."""
    a = _norm_answer(answer)
    ql = (question or "").lower()
    for task, kws in ROUTER_KEYWORDS:
        if any(kw in ql for kw in kws):
            if validate and (task in ANSWER_SETS) and (a not in ANSWER_SETS[task]):
                continue
            return (task, a)
    return None


def parse_qa_labels(qa_field, validate=True):
    """Return {task_name: canonical_answer} for every QA pair we can route.

    Primary path: PulseLM stores qa as a JSON dict keyed by task name
        e.g. {"af_label": {"question":"...","answer":"non_af"},
              "heart_rate_category": {"question":"...","answer":"normal"}}
    We look the key up in TASK_KEY_MAP and use the value's "answer" field.

    Fallback path: list-of-{q,a} dicts, list-of-pairs, single-dict, parallel-list.
    For these we keyword-route on the question text.
    """
    qa = _decode_qa(qa_field)
    if qa is None:
        return {}

    out = {}

    #  Task-keyed dict with {"question":..., "answer":...} values
    if isinstance(qa, dict):
        looks_like_pulselm = any(
            isinstance(v, dict) and ("answer" in v or "a" in v) for v in qa.values()
        )
        if looks_like_pulselm:
            for key, val in qa.items():
                if not isinstance(val, dict):
                    continue
                task = TASK_KEY_MAP.get(str(key).lower())
                ans  = val.get("answer") or val.get("a")
                if ans is None:
                    continue
                a = _norm_answer(ans)
                if task is None:
                    # Unknown task key - try keyword fallback on the question.
                    q = val.get("question") or val.get("q") or ""
                    r = _route_by_question(q, ans, validate)
                    if r is not None:
                        out[r[0]] = r[1]
                    continue
                if validate and (task in ANSWER_SETS) and (a not in ANSWER_SETS[task]):
                    continue
                out[task] = a
            return out

        # Parallel lists: {"question":[...], "answer":[...]}
        qs  = qa.get("question") or qa.get("questions") or qa.get("q")
        as_ = qa.get("answer")   or qa.get("answers")   or qa.get("a")
        if qs is not None and as_ is not None:
            if isinstance(qs, str) and isinstance(as_, str):
                pairs = [(qs, as_)]
            else:
                pairs = list(zip(qs, as_))
            for q, a in pairs:
                r = _route_by_question(str(q), a, validate)
                if r is not None:
                    out[r[0]] = r[1]
        return out

    # Fallback: List of dicts or list of pairs
    if isinstance(qa, (list, tuple)):
        for item in qa:
            if isinstance(item, dict):
                q = item.get("question") or item.get("q") or ""
                a = item.get("answer")   or item.get("a") or ""
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                q, a = item[0], item[1]
            else:
                continue
            r = _route_by_question(str(q), a, validate)
            if r is not None:
                out[r[0]] = r[1]
        return out

    return out


# Test
_TESTS = [
    '{"af_label": {"question": "Is atrial fibrillation present in this recording?", "answer": "non_af"}}',
    '{"heart_rate_category": {"question": "What heart rate classification does this PPG indicate?", "answer": "normal"}, "blood_pressure_category": {"question": "Provide the blood pressure risk category.", "answer": "elevated"}}',
    '[{"question": "Does this PPG signal show atrial fibrillation?", "answer": "af"}, {"question": "What is the heart rate category for this PPG segment?", "answer": "normal"}]',
    '{"weird_arrhythmia_thing": {"question": "What is the arrhythmia category for this segment?", "answer": "pvc"}}',
]
print("Smoke tests:")
for t in _TESTS:
    print(" ", parse_qa_labels(t))


## 1.2: Diagnostic - preview the QA format

Before processing the full dataset, this quick check shows what the QA field looks like in each config and how the parser handles it. It is a simple sanity check that helps catch parsing mistakes early.

In [ ]:
from datasets import load_dataset, get_dataset_config_names
import json as _json

DATASET_ID = "Manhph2211/PulseLM"
_cfgs = get_dataset_config_names(DATASET_ID)

print(f"{'config':<14} {'qa keys':<28} {'parsed labels'}")
print("-" * 96)
for _cfg in _cfgs:
    try:
        _row = next(iter(load_dataset(DATASET_ID, _cfg, split="train", streaming=True)))
    except Exception as _e:
        print(f"{_cfg:<14} ERROR: {type(_e).__name__}: {str(_e)[:60]}")
        continue
    _qa  = _row.get("qa")
    # Describe the structure compactly, then probe one parsed pair
    if isinstance(_qa, str):
        _shape = f"str({len(_qa)})"
        _example_str = _qa[:160].replace("\n", " ")
    elif isinstance(_qa, list) and _qa and isinstance(_qa[0], dict):
        _shape = f"list[{len(_qa)}] of dict {list(_qa[0].keys())}"
        _example_str = None
    elif isinstance(_qa, dict):
        _shape = f"dict {list(_qa.keys())}"
        _example_str = None
    else:
        _shape = type(_qa).__name__
        _example_str = None

    _labels = parse_qa_labels(_qa)
    print(f"{_cfg:<14} {_shape[:28]:<28} {_labels}")
    if _example_str is not None:
        print(f"{'':<14}   raw qa head: {_example_str}")
    # Also pull the first (q, a) the parser sees, if any
    _first = next(iter(_qa_iter(_qa)), None) if callable(globals().get('_qa_iter')) else None
    if _first is not None:
        print(f"{'':<14}   first q: {_first[0][:80]}")
        print(f"{'':<14}   first a: {_first[1][:80]}")


## 2: Streaming ingestion → parquet files with labels

This is where the notebook does the heavy lifting. It streams each dataset split from Hugging Face, extracts the signal features, pulls out the labels, and writes everything to parquet in batches so the process stays manageable.

In [ ]:
import os, time, json, pickle
from datasets import load_dataset, get_dataset_config_names
from tqdm.auto import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

DATASET_ID         = "Manhph2211/PulseLM"
SAMPLES_PER_SPLIT  = 1000         # rows per split
BATCH_SIZE         = 500          # rows per parquet flush
SKIP_EXISTING      = False        # skip configs whose parquet is already on GCS
OUT_DIR            = "gs://egd-us/pulse_features"   #Edit this before running, it must be a GCS path so all workers can read it.

CONFIGS_TO_DROP    = ["afppgecg"]  # too large and imbalanced

config_names = [c for c in get_dataset_config_names(DATASET_ID) if c not in CONFIGS_TO_DROP]
if CONFIGS_TO_DROP:
    print(f"Skipping configs: {CONFIGS_TO_DROP}")
print(f"Using {len(config_names)} configs: {', '.join(config_names)}")

os.makedirs(f"{OUT_DIR}/train",      exist_ok=True)
os.makedirs(f"{OUT_DIR}/validation", exist_ok=True)
os.makedirs(f"{OUT_DIR}/test",       exist_ok=True)

preview_signals = {}

def fetch_to_parquet(cfg, split_name, n_target, batch_size):
    """Stream one (config, split), featurise, parse labels, write batches to parquet.
    Returns row count."""
    try:
        stream = load_dataset(DATASET_ID, cfg, split=split_name, streaming=True)
    except (ValueError, KeyError):
        return 0

    out_path = f"{OUT_DIR}/{split_name}/{cfg}.parquet"
    if SKIP_EXISTING and os.path.exists(out_path):
        # Count rows in the existing file so totals stay accurate.
        try:
            n_existing = pq.ParquetFile(out_path).metadata.num_rows
        except Exception:
            n_existing = 0
        return n_existing
    writer   = None
    batch    = []
    n_written = 0

    pbar = tqdm(desc=f"{cfg}/{split_name}", total=n_target, unit="row", leave=False)

    def _force_label_cols_string(tbl):
        """Pyarrow infers null/int for all-None columns. Force lbl_* to string
        so the parquet schema is stable across configs."""
        new_fields, new_cols = [], []
        for field, col in zip(tbl.schema, tbl.columns):
            if field.name.startswith("lbl_") and not pa.types.is_string(field.type):
                new_fields.append(pa.field(field.name, pa.string()))
                new_cols.append(col.cast(pa.string()))
            else:
                new_fields.append(field)
                new_cols.append(col)
        return pa.Table.from_arrays(new_cols, schema=pa.schema(new_fields))

    def flush():
        nonlocal writer, batch, n_written
        if not batch:
            return
        tbl = pa.Table.from_pylist(batch)
        tbl = _force_label_cols_string(tbl)
        if writer is None:
            writer = pq.ParquetWriter(out_path, tbl.schema, compression="snappy")
        writer.write_table(tbl)
        n_written += len(batch)
        batch.clear()

    try:
        for row in stream:
            if n_target is not None and (n_written + len(batch)) >= n_target:
                break
            signal = row.get("signal")
            if signal is None:
                continue

            # Cache the first ~1000 samples of one signal per config (train split only).
            if split_name == "train" and cfg not in preview_signals:
                preview_signals[cfg] = list(signal[:1000])

            feats = extract_features(signal)
            rec = {f"f_{name}": feats[i] for i, name in enumerate(FEATURE_NAMES)}
            rec["signal_len"] = len(signal)
            rec["text_len"]   = len(row.get("text") or "")
            rec["qa_count"]   = len(row.get("qa") or [])
            rec["dataset"]    = cfg

            qa_labels = parse_qa_labels(row.get("qa"))
            for task in LABEL_TASKS:
                rec[f"lbl_{task}"] = qa_labels.get(task)

            batch.append(rec)
            pbar.update(1)

            if len(batch) >= batch_size:
                flush()
        flush()
    finally:
        if writer is not None:
            writer.close()
        pbar.close()

    return n_written

tasks = [(cfg, sp) for cfg in config_names for sp in ("train", "validation", "test")]
totals = {"train": 0, "validation": 0, "test": 0}

cap = "unlimited" if SAMPLES_PER_SPLIT is None else f"{SAMPLES_PER_SPLIT:,}/split"
print(f"\nIngesting {len(tasks)} (config, split) tasks sequentially (cap: {cap}, batch: {BATCH_SIZE})...\n")
t_start = time.time()

for i, (cfg, sp) in enumerate(tasks, 1):
    try:
        n = fetch_to_parquet(cfg, sp, SAMPLES_PER_SPLIT, BATCH_SIZE)
        totals[sp] += n
        status = f"{n:>6,} rows"
    except Exception as e:
        status = f"FAILED ({type(e).__name__}: {str(e)[:60]})"
    elapsed = time.time() - t_start
    print(f"  [{i:2d}/{len(tasks)}]  {cfg:<14} {sp:<11} {status}   ({elapsed:6.1f}s elapsed)", flush=True)

# save one preview signal per config for the waveform grid
with open(f"{OUT_DIR}/preview_signals.pkl", "wb") as f:
    pickle.dump(preview_signals, f)

print(f"\nDone. train: {totals['train']:,}  |  validation: {totals['validation']:,}  |  test: {totals['test']:,} rows")
print(f"Parquet folder: {OUT_DIR}/  ({len(preview_signals)} preview signals saved)")

## 3: Load features back from disk (lazy, low-memory)

Once the parquet files exist, we can read them back in with pandas for lighter analysis. This is the easy path when you want to inspect the features without spinning up Spark again.

In [ ]:
import pandas as pd

train_pd = pd.read_parquet(f"{OUT_DIR}/train")
val_pd   = pd.read_parquet(f"{OUT_DIR}/validation")
try:
    test_pd = pd.read_parquet(f"{OUT_DIR}/test")
except (FileNotFoundError, OSError):
    test_pd = pd.DataFrame()
    print("(no test parquet found — set SKIP_EXISTING=True in section 2 and re-run that cell to fetch only the test split)")

for name, df in [("train", train_pd), ("val", val_pd), ("test", test_pd)]:
    if len(df):
        mb = df.memory_usage(deep=True).sum() / 1e6
        print(f"{name+'_pd':<10} {len(df):>9,} rows  {mb:>6.1f} MB")

label_cols = [c for c in train_pd.columns if c.startswith("lbl_")]
print(f"\nLabel coverage (rows with non-null label per split):")
cov = pd.DataFrame({
    "train":      train_pd[label_cols].notna().sum(),
    "validation": val_pd[label_cols].notna().sum(),
    "test":       test_pd[label_cols].notna().sum() if len(test_pd) else 0,
})
print(cov.to_string())

train_pd.head()

## 4: Load parquet with Spark for distributed processing

If the dataset is large, Spark makes it easier to work with the parquet files at scale. This section starts a Spark session and reads the saved data back in for the heavier analyses.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os

# On Dataproc the session is already running — reuse it instead of creating a new one.
# Never call .master() here; it would override YARN and drop to local mode.

def _session_is_alive(s):
    try:
        s.sparkContext.applicationId
        return True
    except Exception:
        return False

spark = None

# Step 1 — pre-injected global
_g = globals().get("spark")
if _g is not None and _session_is_alive(_g):
    spark = _g
    print(f"Picked up pre-existing spark global  (master={spark.sparkContext.master})")

# Step 2 — active session registered in the JVM 
if spark is None:
    _a = SparkSession.getActiveSession()
    if _a is not None and _session_is_alive(_a):
        spark = _a
        print(f"Picked up SparkSession.getActiveSession()  (master={spark.sparkContext.master})")

# Step 3 — nothing found: build a new session.
# Only set master=local[*] when we can confirm we are NOT on a managed cluster.
if spark is None:
    _is_managed = (
        os.environ.get("DATAPROC_CLUSTER_NAME") is not None       # Dataproc
        or os.environ.get("DATABRICKS_RUNTIME_VERSION") is not None  # Databricks
        or os.environ.get("EMR_CLUSTER_ID") is not None            # AWS EMR
        or os.path.exists("/etc/spark/conf/spark-defaults.conf")   # any Spark-on-YARN install
    )
    builder = (
        SparkSession.builder
        .appName("pulse_data_v9")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    )
    if not _is_managed:
        builder = builder.master(os.environ.get("SPARK_MASTER", "local[*]"))
        builder = builder.config("spark.driver.memory", "3g")
    spark = builder.getOrCreate()
    print(f"Built new SparkSession  (master={spark.sparkContext.master}, "
          f"managed_detected={_is_managed})")

spark.sparkContext.setLogLevel("WARN")
print(f"\nSpark {spark.version}  |  default parallelism: "
      f"{spark.sparkContext.defaultParallelism}")

# Load parquet
train_df = spark.read.parquet(f"{OUT_DIR}/train")
val_df   = spark.read.parquet(f"{OUT_DIR}/validation")
try:
    test_df = spark.read.parquet(f"{OUT_DIR}/test")
except Exception:
    test_df = None
    print("(no test parquet — re-run section 2 with the test split if needed)")

print(f"train_df: {train_df.count():,} rows")
print(f"val_df:   {val_df.count():,} rows")
if test_df is not None:
    print(f"test_df:  {test_df.count():,} rows")
train_df.printSchema()


## 4.1: Data cleaning: schema check, null/NaN filter, SQI gate

Before we model anything, we make sure the data is usable. That means checking that the expected feature columns exist, dropping rows with missing or invalid values, and filtering out obviously noisy segments using the SQI score.

In [ ]:
rows_per_ds = (
    train_df.groupBy("dataset").count().orderBy(F.desc("count")).toPandas()
)
print(rows_per_ds.to_string(index=False))

In [ ]:
stats_per_ds = (
    train_df.groupBy("dataset")
            .agg(
                F.avg("signal_len").alias("avg_signal_len"),
                F.min("signal_len").alias("min_signal_len"),
                F.max("signal_len").alias("max_signal_len"),
                F.avg("text_len").alias("avg_text_len"),
                F.avg("qa_count").alias("avg_qa_count"),
            )
            .orderBy("dataset")
            .toPandas()
)
stats_per_ds

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.barplot(rows_per_ds, x="count", y="dataset", ax=axes[0, 0], color="#4C78A8")
axes[0, 0].set_title("Rows per sub-dataset (train split)")

sns.barplot(stats_per_ds.sort_values("avg_signal_len", ascending=False),
            x="avg_signal_len", y="dataset", ax=axes[0, 1], color="#F58518")
axes[0, 1].set_title("Average signal length (samples)")

sns.barplot(stats_per_ds.sort_values("avg_text_len", ascending=False),
            x="avg_text_len", y="dataset", ax=axes[1, 0], color="#54A24B")
axes[1, 0].set_title("Average text-description length (chars)")

sns.barplot(stats_per_ds.sort_values("avg_qa_count", ascending=False),
            x="avg_qa_count", y="dataset", ax=axes[1, 1], color="#E45756")
axes[1, 1].set_title("Average #Q&A pairs per sample")

plt.tight_layout(); plt.show()

## 4.2: Label coverage and class distribution

Here we check how much of each dataset is actually labeled for each task, and what the label balance looks like. This helps explain where a model has enough supervision and where it may be working with thin or skewed data.

In [ ]:
# Coverage matrix: one row per config, one column per task, value = fraction of rows labelled.
import pyspark.sql.functions as F

label_cols = [c for c in train_df.columns if c.startswith("lbl_")]

coverage = (
    train_df.groupBy("dataset")
            .agg(*[
                (F.count(F.when(F.col(c).isNotNull(), 1)) / F.count("*")).alias(c)
                for c in label_cols
            ])
            .orderBy("dataset")
            .toPandas()
            .set_index("dataset")
)
coverage.columns = [c.replace("lbl_", "") for c in coverage.columns]
print("Coverage matrix — fraction of train rows in each config that carry each task label:")
print((coverage * 100).round(1).fillna(0).to_string())

In [ ]:
# Class distribution per task
for task in [c.replace("lbl_", "") for c in label_cols]:
    counts = (
        train_df.filter(F.col(f"lbl_{task}").isNotNull())
                .groupBy(f"lbl_{task}")
                .count()
                .orderBy(F.desc("count"))
                .toPandas()
    )
    if len(counts) == 0:
        print(f"\n[{task}] no labels parsed — check ROUTER_KEYWORDS in section 1.5"); continue
    total = counts["count"].sum()
    print(f"\n[{task}] {total:,} labelled rows, {len(counts)} classes:")
    print(counts.assign(pct=lambda d: (d["count"] / total * 100).round(1)).to_string(index=False))

## 4.3: Example signals from each dataset

A quick plot of one sample signal from each config helps confirm that the pipeline is seeing the right kind of waveform. It is a practical visual check before moving on to modeling.

In [ ]:
with open(f"{OUT_DIR}/preview_signals.pkl", "rb") as f:
    preview_signals = pickle.load(f)

samples = sorted(preview_signals.items())
n    = len(samples)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 2.5 * rows))
axes = axes.flatten()

for i, (cfg, sig) in enumerate(samples):
    arr = np.asarray(sig, dtype=float)
    axes[i].plot(arr[:1000], linewidth=0.7)
    axes[i].set_title(cfg, fontsize=10)
    axes[i].set_xticks([]); axes[i].set_yticks([])

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout(); plt.show()

## 5: Machine learning pipeline

Three classifiers trained end-to-end on Spark DataFrames to solve 10 binary clinical prediction tasks. The pipeline demonstrates subset training (labeled rows only), whole-dataset training with class weights, and MLP alternatives. Each section builds diagnostic tables and plots that show per-task accuracy and F1 scores.

### 5.1: Prepare binary target columns
The raw labels are multi-class text strings, but the models in this notebook are set up to answer a simpler question: normal or abnormal. This step converts the labels into that binary form and prepares the matrices we need later.

In [ ]:
import numpy as np
from pyspark.sql import functions as F

ABNORMAL_MAPPING = {
    "HR":         lambda x: x != "normal",
    "BP":         lambda x: x != "normal",
    "Stress":     lambda x: x != "baseline",
    "SDB":        lambda x: not x.startswith("normal"),
    "HRV_SDNN":   lambda x: x != "normal",
    "HRV_RMSSD":  lambda x: x != "normal",
    "HRV_pNN50":  lambda x: x != "normal",
    "AF":         lambda x: x == "af",
    "Arrhythmia": lambda x: x != "sinus_rhythm",
    "RR":         lambda x: x != "normal",
}
ABNORMAL_TASKS = list(ABNORMAL_MAPPING.keys())
feature_cols   = [f"f_{n}" for n in FEATURE_NAMES]


# Spark binary label columns y_HR, y_BP, ... 
def make_binary_label_col(label_col, task):
    c = F.col(label_col)
    if task in ("HR", "BP", "HRV_SDNN", "HRV_RMSSD", "HRV_pNN50", "RR"):
        return F.when(c == "normal", 0).when(c.isNotNull(), 1).otherwise(None)
    if task == "Stress":
        return F.when(c == "baseline", 0).when(c.isNotNull(), 1).otherwise(None)
    if task == "SDB":
        return F.when(c.startswith("normal"), 0).when(c.isNotNull(), 1).otherwise(None)
    if task == "AF":
        return F.when(c == "af", 1).when(c.isNotNull(), 0).otherwise(None)
    if task == "Arrhythmia":
        return F.when(c == "sinus_rhythm", 0).when(c.isNotNull(), 1).otherwise(None)
    raise ValueError(f"No mapping for task {task}")

def add_binary_label_columns(df):
    for task in ABNORMAL_TASKS:
        df = df.withColumn(f"y_{task}", make_binary_label_col(f"lbl_{task}", task))
    return df

train_df_lbl = add_binary_label_columns(train_df).cache()
val_df_lbl   = add_binary_label_columns(val_df).cache()
test_df_lbl  = add_binary_label_columns(test_df).cache() if test_df is not None else None
_ = train_df_lbl.count()


# pandas matrices (used only by the comparison/demo cells for lookups)
def build_multilabel_targets(df):
    Y = np.full((len(df), len(ABNORMAL_TASKS)), np.nan, dtype=np.float32)
    for j, task in enumerate(ABNORMAL_TASKS):
        col = f"lbl_{task}"
        if col not in df.columns: continue
        m = df[col].notna()
        if not m.any(): continue
        Y[m.values, j] = df.loc[m, col].apply(ABNORMAL_MAPPING[task]).astype(np.float32).values
    return Y, ~np.isnan(Y)

Y_train, M_train = build_multilabel_targets(train_pd)
Y_val,   M_val   = build_multilabel_targets(val_pd)
Y_test,  M_test  = build_multilabel_targets(test_pd) if len(test_pd) else (None, None)


print(f"Spark train_df_lbl cached: {train_df_lbl.count():,} rows")
print(f"Pandas Y_train: {Y_train.shape}  ({M_train.sum():,} observed task-labels)")

print(f"\n{'task':<12} {'n_train':>9} {'abn%_train':>11} {'n_val':>8} {'abn%_val':>10} {'n_test':>8} {'abn%_test':>10}")
print("-" * 76)
for j, task in enumerate(ABNORMAL_TASKS):
    def _stats(Y, M):
        if Y is None: return (0, float("nan"))
        n = int(M[:, j].sum())
        rate = float(np.nanmean(Y[:, j])) if n else float("nan")
        return (n, rate)
    n_tr, ab_tr = _stats(Y_train, M_train)
    n_v,  ab_v  = _stats(Y_val,   M_val)
    n_te, ab_te = _stats(Y_test,  M_test)
    print(f"{task:<12} {n_tr:>9,} {ab_tr*100 if n_tr else 0:>10.1f}% "
          f"{n_v:>8,} {ab_v*100 if n_v else 0:>9.1f}% "
          f"{n_te:>8,} {ab_te*100 if n_te else 0:>9.1f}%")

### 5.2: Spark Random Forest - subset training
This version trains one random forest per task, using only the rows that already have a label for that task. It is a straightforward baseline and a good way to see how much signal there is in the features alone.

In [ ]:
from pyspark.ml.feature      import VectorAssembler, StandardScaler as SparkScaler
from pyspark.ml.classification import RandomForestClassifier as SparkRF, MultilayerPerceptronClassifier
from pyspark.ml.evaluation    import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline as SparkPipeline
import time

SP_NUM_TREES    = 50
SP_MAX_DEPTH    = 15
SP_MAX_BINS     = 32
SP_MLP_LAYERS   = [9, 16, 8, 2]
SP_MLP_MAX_ITER = 50
SP_MLP_STEP     = 0.05

_assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
_scaler    = SparkScaler(inputCol="raw_features", outputCol="features",
                          withMean=True, withStd=True)

def _spark_per_task_train(task, df_train, df_val, df_test, model_kind="rf",
                          use_weights=False, sample_cap=None, keep_model=False):
    """Train one binary Spark MLlib classifier on rows where y_{task} is not null.
    model_kind ∈ {"rf", "mlp"}.  If keep_model=True, the fitted Spark Pipeline is
    returned in the result dict so it can be used for prediction later."""
    ycol = f"y_{task}"

    if use_weights:
        wcol = ycol + "_w"
        df_tr = (df_train
                 .withColumn(wcol, F.when(F.col(ycol).isNotNull(), 1.0).otherwise(0.0))
                 .withColumn(ycol, F.when(F.col(ycol).isNotNull(), F.col(ycol)).otherwise(0)))
        weight_col = wcol
    else:
        df_tr = df_train.filter(F.col(ycol).isNotNull())
        weight_col = None

    if sample_cap is not None:
        n_have = df_tr.count()
        if n_have > sample_cap:
            df_tr = df_tr.sample(False, sample_cap / n_have, seed=42)

    n_train = df_tr.count()
    classes = [r[0] for r in df_tr.select(ycol).distinct().collect()]
    if n_train < 200 or len([c for c in classes if c is not None]) < 2:
        return None

    if model_kind == "rf":
        clf = SparkRF(featuresCol="features", labelCol=ycol,
                       numTrees=SP_NUM_TREES, maxDepth=SP_MAX_DEPTH,
                       maxBins=SP_MAX_BINS, seed=42,
                       **({"weightCol": weight_col} if weight_col else {}))
    elif model_kind == "mlp":
        clf = MultilayerPerceptronClassifier(featuresCol="features", labelCol=ycol,
                                              layers=SP_MLP_LAYERS,
                                              maxIter=SP_MLP_MAX_ITER,
                                              stepSize=SP_MLP_STEP, seed=42)
    else:
        raise ValueError(model_kind)

    pipe  = SparkPipeline(stages=[_assembler, _scaler, clf])
    t0    = time.time()
    fitted = pipe.fit(df_tr)
    train_secs = time.time() - t0

    def _eval(df):
        if df is None: return None
        d = df.filter(F.col(ycol).isNotNull())
        if d.rdd.isEmpty(): return None
        pred = fitted.transform(d)
        acc = MulticlassClassificationEvaluator(labelCol=ycol, predictionCol="prediction",
                                                 metricName="accuracy").evaluate(pred)
        f1  = MulticlassClassificationEvaluator(labelCol=ycol, predictionCol="prediction",
                                                 metricName="f1").evaluate(pred)
        return {"n": d.count(), "acc": acc, "f1_macro": f1}

    out = {"task": task, "n_train": n_train, "train_secs": train_secs,
           "val": _eval(df_val), "test": _eval(df_test)}
    if keep_model:
        out["pipeline"] = fitted
    return out


# Section 5.2: per-task RF, subset training
results_sp_rf_subset = {}
print(f"  {'task':<12}  {'n_train':>8}  {'val_acc':>8}  {'val_f1':>7}  {'test_acc':>9}  {'test_f1':>8}  {'secs':>6}")
print("  " + "-" * 64)
for task in ABNORMAL_TASKS:
    r = _spark_per_task_train(task, train_df_lbl, val_df_lbl, test_df_lbl,
                                model_kind="rf", use_weights=False, sample_cap=100_000)
    if r is None:
        print(f"  {task:<12}  [skip — insufficient data]"); continue
    results_sp_rf_subset[task] = r
    va = r["val"] or {}; te = r["test"] or {}
    print(f"  {task:<12}  {r['n_train']:>8,}  "
          f"{va.get('acc', float('nan')):>8.3f}  {va.get('f1_macro', float('nan')):>7.3f}  "
          f"{te.get('acc', float('nan')):>9.3f}  {te.get('f1_macro', float('nan')):>8.3f}  "
          f"{r['train_secs']:>6.1f}")
print(f"\nTrained {len(results_sp_rf_subset)} Spark RFs (subset training).")

### 5.3: Spark Random Forest - whole-dataset training

This version uses the full dataset instead of only the labeled rows. Missing labels are effectively ignored by giving them zero weight, which makes the setup more convenient for comparison while keeping the training logic similar.

In [ ]:
# Whole-dataset Spark RF - also keeps the fitted pipelines so section 5.6 demo can use them.
results_sp_rf_whole = {}
print(f"  {'task':<12}  {'n_train':>9}  {'val_acc':>8}  {'val_f1':>7}  {'test_acc':>9}  {'test_f1':>8}  {'secs':>6}")
print("  " + "-" * 64)
for task in ABNORMAL_TASKS:
    r = _spark_per_task_train(task, train_df_lbl, val_df_lbl, test_df_lbl,
                                model_kind="rf", use_weights=True, sample_cap=None,
                                keep_model=True)
    if r is None:
        print(f"  {task:<12}  [skip — insufficient data]"); continue
    results_sp_rf_whole[task] = r
    va = r["val"] or {}; te = r["test"] or {}
    print(f"  {task:<12}  {r['n_train']:>9,}  "
          f"{va.get('acc', float('nan')):>8.3f}  {va.get('f1_macro', float('nan')):>7.3f}  "
          f"{te.get('acc', float('nan')):>9.3f}  {te.get('f1_macro', float('nan')):>8.3f}  "
          f"{r['train_secs']:>6.1f}")
print(f"\nTrained {len(results_sp_rf_whole)} Spark RFs (whole-dataset with weightCol).")

### 5.4: Spark MLP - multi-layer perceptron per task

This is the neural-network version inside Spark. It is still trained one task at a time, but the model itself can learn a slightly richer decision boundary than the random forest.

In [ ]:
results_sp_mlp = {}
print(f"  {'task':<12}  {'n_train':>8}  {'val_acc':>8}  {'val_f1':>7}  {'test_acc':>9}  {'test_f1':>8}  {'secs':>6}")
print("  " + "-" * 64)
for task in ABNORMAL_TASKS:
    r = _spark_per_task_train(task, train_df_lbl, val_df_lbl, test_df_lbl,
                                model_kind="mlp", use_weights=False, sample_cap=50_000)
    if r is None:
        print(f"  {task:<12}  [skip — insufficient data]"); continue
    results_sp_mlp[task] = r
    va = r["val"] or {}; te = r["test"] or {}
    print(f"  {task:<12}  {r['n_train']:>8,}  "
          f"{va.get('acc', float('nan')):>8.3f}  {va.get('f1_macro', float('nan')):>7.3f}  "
          f"{te.get('acc', float('nan')):>9.3f}  {te.get('f1_macro', float('nan')):>8.3f}  "
          f"{r['train_secs']:>6.1f}")
print(f"\nTrained {len(results_sp_mlp)} Spark MLPs (per-task binary).")

### 5.5: Model comparison and baseline

Now that the models have been trained, this section lines them up side by side on the test set. It also includes a simple majority-class baseline so you can see whether the models are doing anything better than a trivial guess.

In [ ]:
rows = []
sources = [
    ("Spark RF — subset (section 5.2)",   globals().get("results_sp_rf_subset")),
    ("Spark RF — whole (section 5.3)",    globals().get("results_sp_rf_whole")),
    ("Spark MLP (section 5.4)",            globals().get("results_sp_mlp")),
]
present = [(n, d) for n, d in sources if d is not None]
print("Models available:", [n for n, _ in present])

for task in ABNORMAL_TASKS:
    for name, d in present:
        r = d.get(task)
        if not r: continue
        te = r.get("test")
        if not te: continue
        f1 = te.get("f1_macro")
        if f1 is None or (isinstance(f1, float) and np.isnan(f1)): continue
        rows.append({"task": task, "model": name, "test_f1": f1})

if not rows:
    print("No results yet — run section 5.2 / 5.3 / 5.4 first.")
else:
    comp = pd.DataFrame(rows)
    palette = {"Spark RF — subset (section 5.2)": "#4C78A8",
               "Spark RF — whole (section 5.3)":  "#F58518",
               "Spark MLP (section 5.4)":          "#54A24B"}
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.barplot(comp, y="task", x="test_f1", hue="model", ax=ax,
                palette=[palette[m] for m in comp["model"].unique()])
    ax.set_xlim(0, 1); ax.set_title("Per-task test macro-F1 — three Spark classifiers")
    ax.set_xlabel("test macro-F1"); ax.set_ylabel("")
    plt.tight_layout(); plt.show()

    pivot = comp.pivot(index="task", columns="model", values="test_f1")
    print("\nTest macro-F1 table:")
    print(pivot.round(3).fillna("--").to_string())

In [ ]:
# Majority-class baseline
# Lift = model_metric - baseline_metric.
baseline_rows = []
for task in ABNORMAL_TASKS:
    ycol = f"y_{task}"
    tr = train_df_lbl.filter(F.col(ycol).isNotNull())
    te = test_df_lbl.filter(F.col(ycol).isNotNull()) if test_df_lbl is not None else None
    if te is None or te.rdd.isEmpty(): continue

    # Find the majority class on the training set
    maj_row = (tr.groupBy(ycol).count().orderBy(F.desc("count")).first())
    if maj_row is None: continue
    maj_class = int(maj_row[ycol])

    # Apply that constant prediction to the test split
    test_with_pred = te.withColumn("prediction", F.lit(float(maj_class)))
    acc = MulticlassClassificationEvaluator(labelCol=ycol, predictionCol="prediction",
                                              metricName="accuracy").evaluate(test_with_pred)
    f1  = MulticlassClassificationEvaluator(labelCol=ycol, predictionCol="prediction",
                                              metricName="f1").evaluate(test_with_pred)
    baseline_rows.append({
        "task":          task,
        "majority_class": maj_class,
        "majority_pct":   maj_row["count"] / tr.count(),
        "baseline_acc":  acc,
        "baseline_f1":   f1,
    })

baseline_df = pd.DataFrame(baseline_rows).set_index("task")

# Compare against the best model's test F1 per task (whichever of the three is highest)
def _best_test_f1(task):
    best = None
    for d in [results_sp_rf_subset, results_sp_rf_whole, results_sp_mlp]:
        r = (d or {}).get(task)
        if r and r.get("test") and r["test"].get("f1_macro") is not None:
            f1 = r["test"]["f1_macro"]
            if best is None or f1 > best:
                best = f1
    return best

baseline_df["best_model_f1"] = [_best_test_f1(t) for t in baseline_df.index]
baseline_df["lift"]          = baseline_df["best_model_f1"] - baseline_df["baseline_f1"]

print("Majority-class baseline vs best Spark model (test macro-F1):")
print(baseline_df.round(3).to_string())

# Plot - bars showing baseline (gray) vs best model (green), with lift annotated
fig, ax = plt.subplots(figsize=(11, 4))
xs = range(len(baseline_df))
ax.barh(xs, baseline_df["baseline_f1"], color="#bbbbbb", label="majority-class baseline")
ax.barh(xs, baseline_df["best_model_f1"] - baseline_df["baseline_f1"],
        left=baseline_df["baseline_f1"], color="#54A24B", label="model lift")
ax.set_yticks(list(xs)); ax.set_yticklabels(baseline_df.index)
ax.set_xlabel("test macro-F1"); ax.set_xlim(0, 1)
ax.set_title("Best Spark model vs always-predict-majority baseline")
ax.legend(); plt.tight_layout(); plt.show()

### 5.6: Run clinical predictions on test signals

This last section applies the trained network to real test examples and prints the predictions in a readable way. It is the closest thing in the notebook to a worked example of how the model behaves on actual signals.

In [ ]:
from pyspark.sql.types import Row

TASK_QUESTIONS = {
    "HR":         "Is the heart rate abnormal (brady- or tachycardia)?",
    "BP":         "Is the blood pressure abnormal (elevated / hypertensive)?",
    "Stress":     "Is the subject in a non-baseline emotional state?",
    "SDB":        "Does the patient have any sleep-disordered breathing (apnea)?",
    "HRV_SDNN":   "Is the HRV (SDNN) abnormal?",
    "HRV_RMSSD":  "Is the HRV (RMSSD) abnormal?",
    "HRV_pNN50":  "Is the HRV (pNN50) abnormal?",
    "AF":         "Does this signal show atrial fibrillation?",
    "Arrhythmia": "Does this signal show any arrhythmia (vs. sinus rhythm)?",
    "RR":         "Is the respiratory rate abnormal (brady- or tachypnea)?",
}


def predict_clinical_spark(row_df, threshold=0.5):
    """row_df: a 1-row Spark DataFrame with feature columns + any y_* placeholders.
    Returns {task: (yes/no, probability)} using the section 5.3 whole-dataset RF pipelines."""
    out = {}
    for task in ABNORMAL_TASKS:
        r = results_sp_rf_whole.get(task)
        if not r or "pipeline" not in r:
            out[task] = (None, None); continue
        pipe = r["pipeline"]
        ycol = f"y_{task}"
        # Pipeline needs the y_{task} column present since we trained with the label column. Add 0 as a placeholder if missing.
        in_df = row_df
        if ycol not in in_df.columns:
            in_df = in_df.withColumn(ycol, F.lit(0))
        pred = pipe.transform(in_df).select("prediction", "probability").collect()[0]
        # probability is a Spark DenseVector; index 1 is P(class=1)=abnormal
        prob_abn = float(pred["probability"][1])
        out[task] = (prob_abn >= threshold, prob_abn)
    return out


def explain_row_spark(pos_idx, df=test_pd):
    # Build a 1-row Spark DataFrame from the pandas test row.
    row = df.iloc[[pos_idx]].copy()
    # Add empty y_* columns so the pipeline's VectorAssembler finds all expected fields
    for task in ABNORMAL_TASKS:
        if f"y_{task}" not in row.columns:
            row[f"y_{task}"] = 0
    row_sdf = spark.createDataFrame(row[feature_cols + [f"y_{t}" for t in ABNORMAL_TASKS]])

    pred = predict_clinical_spark(row_sdf)
    print(f"\n=== Row {pos_idx}  (dataset={df.iloc[pos_idx]['dataset']}) ===")
    print(f"{'question':<60} {'pred':>8} {'p(abn)':>8}  ground truth")
    print("-" * 100)
    for task in ABNORMAL_TASKS:
        yn, p = pred[task]
        if yn is None:
            print(f"{TASK_QUESTIONS[task]:<60} {'--':>8} {'--':>8}  [no model]")
            continue
        col   = f"lbl_{task}"
        truth_raw = df.iloc[pos_idx].get(col)
        if pd.isna(truth_raw):
            truth_str = "--"; mark = " "
        else:
            truth_is_abn = ABNORMAL_MAPPING[task](truth_raw)
            truth_str = f"{'Yes' if truth_is_abn else 'No':<3}  ({truth_raw})"
            mark = "✓" if truth_is_abn == yn else "✗"
        print(f"{TASK_QUESTIONS[task]:<60} {'Yes' if yn else 'No':>8} {p:>8.2f}  {mark} {truth_str}")


sample_rows = []
for cfg in ["ppgarrhythmia", "sdb", "wesad", "uci", "vitaldb"]:
    cand = test_pd[test_pd["dataset"] == cfg]
    if len(cand) > 0:
        sample_rows.append(test_pd.index.get_loc(cand.index[0]))
sample_rows = sample_rows[:4]

for pos_idx in sample_rows:
    explain_row_spark(pos_idx)

### 5.7: Runtime benchmark - per-phase timings

Times three phases independently and saves results to `runtime_<CLUSTER_LABEL>.json`. Run this on each cluster size (2w / 4w) to populate the cross-cluster scaling plot.

- **EDA** - distributed groupBy-aggregate
- **Preprocessing** - parquet read + filter + repartition
- **Training** - sum of section 5.3 wall-clock times across all tasks

Set `CLUSTER_LABEL` (or the env var) before running. The plot at the end renders speedup curves per phase against a linear-ideal reference.

In [ ]:
import os, glob, json, datetime, time
import pandas as pd

# A. Set a label for THIS cluster size. Change between runs.
CLUSTER_LABEL = os.environ.get("CLUSTER_LABEL", "dataproc_4w")

# B. Capture cluster metadata from Spark
sc = spark.sparkContext
import subprocess, time as _time

def _get_worker_count_yarn():
    """Ask YARN directly how many NodeManagers are active. Most reliable on Dataproc."""
    try:
        out = subprocess.run(
            ["yarn", "node", "-list", "-states", "RUNNING"],
            capture_output=True, text=True, timeout=10
        )
        # Each running node appears as a line starting with a hostname
        nodes = [l for l in out.stdout.splitlines()
                 if l.strip() and not l.startswith("Total") and ":" in l
                 and not l.startswith("RUNNING") and not l.startswith("Node")]
        if nodes:
            return len(nodes)
    except Exception:
        pass
    return None

def _get_executor_info():
    """
    Detect live executor count. Strategies tried in order:
    1. YARN NodeManager list - ground truth, always right on Dataproc
    2. statusTracker - accurate after a prior Spark action
    3. defaultParallelism / cores_per_executor - reliable estimate
    4. spark.executor.instances conf - static allocation only
    """
    cores_per_exec = int(sc.getConf().get("spark.executor.cores") or 1)

    # Strategy 1: YARN
    yarn_workers = _get_worker_count_yarn()
    if yarn_workers:
        total = yarn_workers * cores_per_exec if cores_per_exec else sc.defaultParallelism
        return yarn_workers, total

    # Strategy 2: statusTracker
    try:
        infos = sc._jsc.sc().statusTracker().getExecutorInfos()
        execs = [i for i in infos if str(i.executorId()) != "driver"]
        if execs:
            return len(execs), sum(int(i.totalCores()) for i in execs)
    except Exception:
        pass

    # Strategy 3: defaultParallelism / cores_per_executor
    dpar = sc.defaultParallelism
    if dpar and cores_per_exec:
        return max(1, dpar // cores_per_exec), dpar

    # Strategy 4: static allocation
    static = sc.getConf().get("spark.executor.instances")
    if static and int(static) > 0:
        n = int(static)
        return n, n * cores_per_exec

    return 0, sc.defaultParallelism or 0

# Warm-up: enough partitions to force YARN to allocate all executor slots
_warmup_parts = max(100, sc.defaultParallelism * 8)
sc.parallelize(range(_warmup_parts), _warmup_parts).map(lambda x: x * x).sum()
_time.sleep(2) 

n_executors, total_cores = _get_executor_info()
print(f"  Executor detection: n_executors={n_executors}, total_cores={total_cores}, "
      f"defaultParallelism={sc.defaultParallelism}")
print(f"  (spark.executor.cores={sc.getConf().get('spark.executor.cores') or 'unset'}, "
      f"spark.executor.instances={sc.getConf().get('spark.executor.instances') or 'unset'})")
cluster_info = {
    "label":               CLUSTER_LABEL,
    "spark_version":       spark.version,
    "n_executors":         n_executors,
    "total_executor_cores": total_cores,
    "default_parallelism": sc.defaultParallelism,
    "captured_at":         datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
}
print("Cluster metadata for this run:")
for k, v in cluster_info.items():
    print(f"  {k:<22} {v}")

# C. Time three distinct phases independently
phases = {}

# Phase 1 - distributed EDA aggregation (per-class mean of every feature).
print("\n[Phase 1] EDA aggregation ...")
t0 = time.perf_counter()
_ = (train_df_lbl
     .groupBy("dataset")
     .agg(*[F.avg(c).alias(c) for c in feature_cols])
     .collect())  # .collect() forces real computation, .count() does not
phases["eda_s"] = round(time.perf_counter() - t0, 3)
print(f"  {phases['eda_s']:.2f}s")

# Phase 2 - preprocessing: re-read parquet, drop nulls, repartition.
print("\n[Phase 2] Preprocessing (read + filter + repartition) ...")
t0 = time.perf_counter()
_tmp = spark.read.parquet(f"{OUT_DIR}/train")
_tmp = _tmp.filter(F.col("f_mean").isNotNull())
_tmp = _tmp.repartition(max(2, n_executors * 4))
_ = _tmp.count()
phases["prep_s"] = round(time.perf_counter() - t0, 3)
print(f"  {phases['prep_s']:.2f}s")

# Phase 3 - already measured during section 5.3 training. Sum across tasks.
phases["train_s"] = round(
    sum(r["train_secs"] for r in (results_sp_rf_whole or {}).values()), 3
)
print(f"\n[Phase 3] Training (sum of section 5.3 train_secs): {phases['train_s']:.2f}s")

phases["total_s"] = round(sum(phases.values()) - phases.get("total_s", 0), 3)
print(f"\n  TOTAL: {phases['total_s']:.2f}s")

# D. Save this run to JSON
record = {**cluster_info, "phases": phases}
bench_dir = f"{OUT_DIR}/benchmarks" if not OUT_DIR.startswith("gs://") else "./benchmarks"
os.makedirs(bench_dir, exist_ok=True)
json_path = f"{bench_dir}/runtime_{CLUSTER_LABEL}.json"
with open(json_path, "w") as f:
    json.dump(record, f, indent=2)
print(f"\nSaved → {json_path}")

# If OUT_DIR is GCS, also push the JSON to GCS so it's visible across clusters.
if OUT_DIR.startswith("gs://"):
    import subprocess
    gcs_path = f"{OUT_DIR}/benchmarks/runtime_{CLUSTER_LABEL}.json"
    subprocess.run(["gsutil", "cp", json_path, gcs_path], check=False)
    print(f"Pushed → {gcs_path}")

# E. Cross-run scaling plot (if multiple JSONs exist)
# Pull all benchmark JSONs (local first, GCS if available)
all_records = []
for jp in sorted(glob.glob(f"{bench_dir}/runtime_*.json")):
    with open(jp) as f:
        all_records.append(json.load(f))

if OUT_DIR.startswith("gs://"):
    import subprocess
    try:
        out = subprocess.run(["gsutil", "ls", f"{OUT_DIR}/benchmarks/runtime_*.json"],
                              capture_output=True, text=True, check=False)
        for gcs_p in out.stdout.strip().split("\n"):
            if not gcs_p: continue
            local_p = f"/tmp/{os.path.basename(gcs_p)}"
            subprocess.run(["gsutil", "cp", gcs_p, local_p], check=False)
            try:
                with open(local_p) as f:
                    rec = json.load(f)
                    if rec not in all_records:
                        all_records.append(rec)
            except Exception: pass
    except Exception: pass

print(f"\nFound {len(all_records)} benchmark run(s).")

if len(all_records) > 1:
    df_runs = pd.DataFrame([{**{k: v for k, v in r.items() if k != "phases"},
                              **r["phases"]} for r in all_records])
    df_runs = df_runs.sort_values("n_executors")
    print("\nCross-run timing table:")
    print(df_runs[["label", "n_executors", "eda_s", "prep_s", "train_s", "total_s"]]
          .round(2).to_string(index=False))

    # Stacked-bar absolute time + speedup curve
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: stacked bars per cluster
    xs = range(len(df_runs))
    colors = {"eda_s": "#4C78A8", "prep_s": "#F58518", "train_s": "#54A24B"}
    bottom = [0]*len(df_runs)
    for ph, col in colors.items():
        vals = df_runs[ph].tolist()
        axes[0].bar(xs, vals, bottom=bottom, color=col, label=ph.replace("_s",""), edgecolor="white")
        bottom = [b+v for b, v in zip(bottom, vals)]
    axes[0].set_xticks(xs); axes[0].set_xticklabels(df_runs["label"], rotation=20)
    axes[0].set_ylabel("Wall-clock time (s)"); axes[0].set_title("Per-phase time per cluster")
    axes[0].legend()

    # Right: speedup curve (relative to smallest cluster)
    base = df_runs.iloc[0]
    baseline_execs = base["n_executors"]
    # x-axis: prefer n_executors from statusTracker; fall back to
    # total_executor_cores / cores_per_exec (always reliable on Dataproc).
    def _x_axis(row):
        if row["n_executors"] > 0:
            return row["n_executors"]
        cores_per = int(sc.getConf().get("spark.executor.cores") or 4)
        return max(1, row["total_executor_cores"] // cores_per)
    df_runs["n_exec_plot"] = df_runs.apply(_x_axis, axis=1)
    baseline_execs_plot = df_runs.iloc[0]["n_exec_plot"]

    for ph, col in colors.items():
        speedups = base[ph] / df_runs[ph]
        axes[1].plot(df_runs["n_exec_plot"], speedups, "o-", color=col,
                     markersize=10, linewidth=2, label=ph.replace("_s",""))
    # ideal-linear reference
    ideal_x = df_runs["n_exec_plot"]
    axes[1].plot(ideal_x, ideal_x / baseline_execs_plot,
                 "k--", alpha=0.4, label="linear ideal")
    axes[1].set_xlabel("Number of Spark executors")
    axes[1].set_ylabel(f"Speedup (× over {int(baseline_execs_plot)} executors)")
    axes[1].set_title("Per-phase speedup vs cluster size")
    axes[1].grid(alpha=0.3); axes[1].legend()
    plt.tight_layout(); plt.show()
else:
    print("(Run this cell on different cluster sizes to populate the scaling plot.)")